# Train Chess Figurine Classifier (HOG + MLP → TFLite)

Train a 5-class classifier to recognize chess piece figurines (K, Q, R, B, N)
using **HOG (Histogram of Oriented Gradients) features** fed into a small **Keras MLP**,
exported to TFLite.

**Why HOG instead of raw pixels?**
HOG captures gradient structure (edges, shapes) that distinguishes K/Q/R/B/N regardless
of font, DPI, or rendering style — much more robust than pixel-level CNN features.

**Why a custom HOG (not scikit-image)?**
The HOG computation mirrors `HogExtractor.extract()` in Flutter **exactly** —
no rounding differences, no library version drift.

**Why ±35° rotation augmentation?**
PDF pages are sometimes scanned at a slight angle (up to ~30°). Augmenting with ±35°
makes the model robust to tilted glyphs.

**Input:** `chess_glyphs_classifier.zip` from `extract_chess_glyphs`, uploaded to Drive.  
**Output:** `figurine_classifier.tflite` — model input is a 1767-float HOG vector.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

## Step 2 — Copy zip from Drive and extract locally

Upload `chess_glyphs_classifier.zip` (produced by `extract_chess_glyphs`) to Google Drive
**as a single zip file** — do not unzip it on your machine first.  
Set `ZIP_ON_DRIVE` to its path, then run this cell.  
The zip is copied to Colab's local disk and extracted there for fast I/O during training.

In [ ]:
import os, shutil, zipfile

# ── EDIT THIS: path to the zip on your Drive ──────────────────────────────
ZIP_ON_DRIVE = '/content/gdrive/MyDrive/entrainement_ocr_echecs/chess_glyphs_classifier.zip'

LOCAL_ZIP    = '/content/chess_glyphs_classifier.zip'
EXTRACT_DIR  = '/content/glyphs_extracted'

if not os.path.exists(ZIP_ON_DRIVE):
    raise FileNotFoundError(
        f'Zip not found on Drive: {ZIP_ON_DRIVE}\n'
        f'Upload chess_glyphs_classifier.zip to that location first.'
    )

# Copy zip to local disk (fast — stays on Colab's own storage)
print('Copying zip from Drive to local disk...')
shutil.copy2(ZIP_ON_DRIVE, LOCAL_ZIP)
size_mb = os.path.getsize(LOCAL_ZIP) / (1024 * 1024)
print(f'  Copied: {size_mb:.1f} MB')

# Extract
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
print('Extracting...')
with zipfile.ZipFile(LOCAL_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Locate glyphs/ folder — search in root and one level down
GLYPHS_DIR = None
for candidate in [
    os.path.join(EXTRACT_DIR, 'glyphs'),
    os.path.join(EXTRACT_DIR, 'glyphs_results', 'glyphs'),
]:
    if os.path.exists(candidate):
        GLYPHS_DIR = candidate
        break

if GLYPHS_DIR is None:
    raise FileNotFoundError(
        f'Could not locate glyphs/ folder inside the zip.\n'
        f'Top-level contents: {os.listdir(EXTRACT_DIR)}'
    )

print(f'\n✅ Glyphs extracted to {GLYPHS_DIR}')
from PIL import Image

def count_images(folder):
    count = 0
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        if not os.path.isfile(path):
            continue
        try:
            Image.open(path).verify()
            count += 1
        except Exception:
            pass
    return count

total = 0
for piece in sorted(os.listdir(GLYPHS_DIR)):
    piece_dir = os.path.join(GLYPHS_DIR, piece)
    if not os.path.isdir(piece_dir):
        continue
    n = count_images(piece_dir)
    total += n
    print(f'  {"✅" if n > 0 else "❌"} {piece}: {n} images')
print(f'  Total: {total} images')

## Step 2.5 — Extract figurines from multiple PDFs

Extract positive samples (K, Q, R, B, N figurines) from multiple PDF books with different
figurine notation styles. The figurine detector in stage 1 will handle filtering, so we only
need to classify piece types here.

Set `FIGURINE_PDFS` to a list of PDFs containing figurine notation.

In [ ]:
!apt-get install -y -q poppler-utils
!pip install -q pdfplumber pdf2image

import os, random, pdfplumber, shutil
from pdf2image import convert_from_path
from PIL import Image
from collections import defaultdict

# ── EDIT THESE ────────────────────────────────────────────────────────────
FIGURINE_PDFS = [
    '/content/gdrive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/CommentDevenirSuperAttaquant.pdf',
    '/content/gdrive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/ChessTrategy_Grivas_1.pdf',
    '/content/gdrive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/BoussoleSurEchiquier.pdf',
]

SKIP_FIGURINE_EXTRACTION = False  # set True to skip figurine extraction (use zip only)

FIG_PAGES_START         = 1       # first page to scan for figurines
FIG_PAGES_END           = 999     # last page to scan
FIG_MIN_SIZE_PT         = 12.0    # must match Flutter FigurineDetector.minGlyphSize
FIG_MAX_ASPECT          = 2.0     # must match Flutter FigurineDetector.maxAspectRatio
FIG_DPI                 = 150     # render DPI
# ──────────────────────────────────────────────────────────────────────────

def extract_candidates_from_pdf(pdf_path, pages_start, pages_end, min_size_pt, max_aspect, dpi):
    """
    Extract all character candidates from a PDF that pass size/aspect filters.
    Returns list of (crop_pil_image, page_num) tuples.
    """
    candidates = []

    if not os.path.exists(pdf_path):
        print(f'⚠️  PDF not found: {pdf_path}')
        return candidates

    scale = dpi / 72.0

    with pdfplumber.open(pdf_path) as pdf:
        n_pages = len(pdf.pages)
        p_end = min(pages_end, n_pages)
        print(f'  Scanning pages {pages_start}–{p_end} of {n_pages} at {dpi} DPI…')

        for page_idx in range(pages_start - 1, p_end):
            page_num = page_idx + 1
            page = pdf.pages[page_idx]

            imgs = convert_from_path(
                pdf_path,
                first_page=page_num, last_page=page_num,
                dpi=dpi,
            )
            if not imgs:
                continue
            page_img = imgs[0]

            page_found = 0
            for ch in (page.chars or []):
                w_pt = ch.get('width', 0.0)
                h_pt = ch.get('height', 0.0)
                if w_pt < min_size_pt or h_pt < min_size_pt:
                    continue
                ar = w_pt / max(h_pt, 1e-6)
                if ar < 1.0 / max_aspect or ar > max_aspect:
                    continue

                x0 = max(0, int(ch['x0'] * scale))
                y0 = max(0, int(ch['top'] * scale))
                x1 = min(page_img.width, int(ch['x1'] * scale))
                y1 = min(page_img.height, int(ch['bottom'] * scale))
                if x1 <= x0 or y1 <= y0:
                    continue

                crop = page_img.crop((x0, y0, x1, y1))
                if crop.width < 4 or crop.height < 4:
                    continue
                candidates.append((crop, page_num))
                page_found += 1

            if page_found:
                print(f'    Page {page_num:3d}: {page_found} candidates  (total {len(candidates)})', flush=True)

    return candidates

# ─────────────────────────────────────────────────────────────────────────
# Extract figurines from multiple PDFs
# ─────────────────────────────────────────────────────────────────────────

if SKIP_FIGURINE_EXTRACTION:
    print('⏭️  Skipping figurine extraction')
else:
    print(f'\n{"="*70}')
    print('EXTRACTING FIGURINES FROM PDFs')
    print(f'{"="*70}\n')

    all_candidates = []
    for pdf_path in FIGURINE_PDFS:
        if not os.path.exists(pdf_path):
            print(f'⚠️  PDF not found: {pdf_path}')
            continue
        print(f'Extracting from: {os.path.basename(pdf_path)}')
        candidates = extract_candidates_from_pdf(
            pdf_path, FIG_PAGES_START, FIG_PAGES_END,
            FIG_MIN_SIZE_PT, FIG_MAX_ASPECT, FIG_DPI
        )
        all_candidates.extend(candidates)
        print(f'  → {len(candidates)} candidates')

    print(f'\n📊 Total candidates from all figurine PDFs: {len(all_candidates)}')

    if all_candidates:
        temp_fig_dir = os.path.join(GLYPHS_DIR, '_figurine_candidates_unclassified')
        os.makedirs(temp_fig_dir, exist_ok=True)

        for i, (crop, page_num) in enumerate(all_candidates):
            crop.save(os.path.join(temp_fig_dir, f'{i+1:06d}_p{page_num}.png'))

        print(f'\n✅ Saved {len(all_candidates)} unclassified figurine candidates to:')
        print(f'   {temp_fig_dir}')
        print('   → Manually review and sort these into K/, Q/, R/, B/, N/ folders,')
        print('     or use the classification step below.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# OPTIONAL: Cluster figurine candidates using HOG distance (unsupervised)
# ─────────────────────────────────────────────────────────────────────────

CLASSIFY_UNCLASSIFIED_FIGURINES = False  # set True to attempt automatic clustering

if CLASSIFY_UNCLASSIFIED_FIGURINES:
    temp_fig_dir = os.path.join(GLYPHS_DIR, '_figurine_candidates_unclassified')
    
    if os.path.exists(temp_fig_dir):
        from sklearn.cluster import KMeans
        
        print(f'\nClustering unclassified figurine candidates into 5 piece types...')
        
        # Load all unclassified images
        fig_files = sorted([f for f in os.listdir(temp_fig_dir) if f.endswith('.png')])
        if not fig_files:
            print('  No unclassified figurine candidates found.')
        else:
            fig_images = []
            fig_features = []
            
            for fname in fig_files:
                fpath = os.path.join(temp_fig_dir, fname)
                with Image.open(fpath) as img:
                    img_gray = img.convert('L').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
                    arr = np.array(img_gray, dtype=np.float32) / 255.0
                    
                    # Compute HOG
                    hog_vec = _compute_hog_features(arr)
                    fig_features.append(hog_vec)
                    fig_images.append((fname, fpath, arr.shape[0], arr.shape[1]))
            
            fig_features = np.array(fig_features, dtype=np.float32)
            
            # Cluster into K=5 groups (K, Q, R, B, N)
            n_clusters = min(5, len(fig_files))
            print(f'  {len(fig_files)} candidates, clustering into {n_clusters} groups...')
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            labels = kmeans.fit_predict(fig_features)
            
            # Assign clusters to piece types by size (largest cluster first)
            cluster_counts = defaultdict(int)
            for label in labels:
                cluster_counts[label] += 1
            
            sorted_clusters = sorted(cluster_counts.items(), key=lambda x: -x[1])
            cluster_to_class = {}
            for cluster_idx, (clust, _) in enumerate(sorted_clusters):
                cluster_to_class[clust] = CLASS_NAMES[cluster_idx]
            
            # Move/copy files to class folders
            print('\n  Organizing by cluster:')
            for cluster_label in sorted(set(labels)):
                class_name = cluster_to_class[cluster_label]
                class_dir = os.path.join(GLYPHS_DIR, class_name)
                os.makedirs(class_dir, exist_ok=True)
                
                mask = labels == cluster_label
                count = mask.sum()
                
                for i, (fname, fpath, _, _) in enumerate(fig_images):
                    if mask[i]:
                        dst_path = os.path.join(class_dir, fname.replace('_p', f'_c{cluster_label}_p'))
                        shutil.copy2(fpath, dst_path)
                
                print(f'    Cluster {cluster_label} → {class_name}: {count} images')
            
            print(f'\n✅ Organized {len(fig_files)} figurine candidates into class folders')
            print('   ⚠️  Review the results and manually adjust if needed.')
    else:
        print(f'⚠️  No unclassified figurine candidates found at {temp_fig_dir}')
        print('   Did you run Step 2.5 with SKIP_FIGURINE_EXTRACTION=False?')

## Step 2.6 — Classify figurine candidates (optional)

If you extracted figurine candidates from multiple PDFs in Step 2.5, this step helps organize
them into K/, Q/, R/, B/, N/ folders using visual clustering or manual review.

**Option 1 (automatic clustering):** Group visually similar candidates using HOG distance.
**Option 2 (manual):** Review `_figurine_candidates_unclassified` folder and manually sort into class folders.

## Step 3 — Install dependencies

In [ ]:
!pip install -q tensorflow pillow numpy scikit-image scikit-learn matplotlib

## Step 4 — Configuration and HOG feature extractor

**`_compute_hog_features` is the reference implementation.**  
It mirrors `HogExtractor.extract()` in `lib/chess/hog_extractor.dart` line-for-line.  
Do **not** replace this with `skimage.feature.hog` — scikit-image's internals differ
subtly and would break training/inference consistency.

In [ ]:
import gc
import ctypes
import random
import numpy as np
from PIL import Image, ImageOps, ImageFilter

CLASS_NAMES           = ['K', 'Q', 'R', 'B', 'N']
IMG_SIZE              = 32
MAX_ROTATION_DEG      = 35
TARGET_TOTAL_SAMPLES  = 150_000
EPOCHS                = 80
BATCH_SIZE            = 128
VAL_SPLIT             = 0.15
TFLITE_PATH           = 'figurine_classifier.tflite'

# HOG constants — must stay in sync with hog_extractor.dart
_ORIENTATIONS = 9
_PX_PER_CELL  = 4
_CPB          = 2
_N_CELLS      = IMG_SIZE // _PX_PER_CELL          # 8
_N_BLOCKS     = _N_CELLS - _CPB + 1               # 7
_BLOCK_SIZE   = _CPB * _CPB * _ORIENTATIONS       # 36
FEATURE_DIM   = _N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE + 3  # 1767

# Precompute pixel→cell mapping once
_CY   = np.repeat(np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_CX   = np.tile(  np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_BASE = (_CY * _N_CELLS + _CX) * _ORIENTATIONS

# malloc_trim: tell glibc to return freed memory to the OS (Linux/Colab only)
try:
    _libc = ctypes.CDLL('libc.so.6')
    _has_malloc_trim = True
    print('malloc_trim available ✅')
except Exception:
    _has_malloc_trim = False
    print('malloc_trim not available (non-Linux), GC only')


def _malloc_trim():
    gc.collect()
    if _has_malloc_trim:
        _libc.malloc_trim(0)


def _compute_hog_features(img_32x32: np.ndarray) -> np.ndarray:
    """Vectorized HOG — matches Dart HogExtractor.extract() exactly."""
    gx = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gy = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gx[:, 1:-1] = img_32x32[:, 2:] - img_32x32[:, :-2]
    gy[1:-1, :] = img_32x32[2:, :] - img_32x32[:-2, :]

    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = np.degrees(np.arctan2(gy, gx)) % 180.0

    bin_width = 180.0 / _ORIENTATIONS
    bf = ang.ravel() / bin_width
    b0 = bf.astype(np.int32) % _ORIENTATIONS
    b1 = (b0 + 1) % _ORIENTATIONS
    t  = bf - b0
    mf = mag.ravel()

    flat = np.bincount(
        np.concatenate([_BASE + b0, _BASE + b1]),
        weights=np.concatenate([mf * (1.0 - t), mf * t]),
        minlength=_N_CELLS * _N_CELLS * _ORIENTATIONS,
    )
    cell_hists = flat.reshape(_N_CELLS, _N_CELLS, _ORIENTATIONS)

    eps2    = 1e-5 ** 2
    hog_out = np.empty(_N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE, dtype=np.float64)
    out_i   = 0
    for by in range(_N_BLOCKS):
        for bx in range(_N_BLOCKS):
            block = cell_hists[by:by + _CPB, bx:bx + _CPB, :].ravel().copy()
            block /= np.sqrt(np.dot(block, block) + eps2)
            np.clip(block, 0.0, 0.2, out=block)
            block /= np.sqrt(np.dot(block, block) + eps2)
            hog_out[out_i:out_i + _BLOCK_SIZE] = block
            out_i += _BLOCK_SIZE
    return hog_out


def extract_features(arr_32x32: np.ndarray, orig_w: int, orig_h: int) -> np.ndarray:
    """
    HOG + shape stats from a pre-resized 32×32 float32 numpy array.
    No PIL, no skimage — zero extra allocation beyond the HOG itself.
    orig_w / orig_h: original glyph pixel dimensions (for aspect ratio feature).
    """
    img_gray = arr_32x32.astype(np.float64)
    hog_feat = _compute_hog_features(img_gray)
    extra = np.array([
        orig_w / max(orig_h, 1),
        np.mean(img_gray),
        np.std(img_gray),
    ], dtype=np.float64)
    return np.concatenate([hog_feat, extra]).astype(np.float32)


# Sanity-check
_dummy = extract_features(np.full((IMG_SIZE, IMG_SIZE), 0.5, dtype=np.float32), 32, 32)
assert len(_dummy) == FEATURE_DIM
del _dummy
mem_gb = TARGET_TOTAL_SAMPLES * FEATURE_DIM * 4 / 1e9
print(f'HOG feature vector : {FEATURE_DIM} dims  ✅')
print(f'Target samples     : {TARGET_TOTAL_SAMPLES:,}  (X ≈ {mem_gb:.2f} GB pre-allocated)')
print(f'Classes            : {len(CLASS_NAMES)}  {CLASS_NAMES}')

## Step 5 — Index labeled glyph images (paths only, no RAM cost)

In [ ]:
from collections import defaultdict

def index_glyphs(glyphs_dir, class_names):
    """Return (paths, labels, counts) — images are NOT loaded into RAM."""
    paths, labels, counts = [], [], defaultdict(int)
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(glyphs_dir, class_name)
        if not os.path.exists(class_dir):
            print(f'⚠️  {class_name} folder not found')
            continue
        for filename in sorted(os.listdir(class_dir)):
            filepath = os.path.join(class_dir, filename)
            if os.path.isfile(filepath) and filename.lower().endswith('.png'):
                paths.append(filepath)
                labels.append(class_idx)
                counts[class_name] += 1
    return paths, labels, counts

print(f'Indexing glyphs in {GLYPHS_DIR}...\n')
paths, labels, counts = index_glyphs(GLYPHS_DIR, CLASS_NAMES)
print(f'✅ Indexed {len(paths)} glyph images (paths only — no RAM used)\n')
for name in CLASS_NAMES:
    n = counts[name]
    print(f'  {"✅" if n > 0 else "❌"} {name}: {n}')

## Step 6 — Data augmentation + HOG feature extraction

In [ ]:
def augment_to_array(pil_img, n, size=32):
    """
    Augment a PIL image n times.
    Returns a list of float32 (size, size) numpy arrays — no PIL objects kept alive.
    Immediately converts each PIL result to numpy and discards the PIL object.
    """
    orig_w, orig_h = pil_img.width, pil_img.height
    # Resize once to 32×32 in numpy — all augmentation happens at this scale
    base_pil = pil_img.convert('L').resize((size, size), Image.LANCZOS)
    base = np.array(base_pil, dtype=np.float32) / 255.0
    del base_pil

    results = []
    for _ in range(n):
        arr = base.copy()

        # Rotation (PIL, converted immediately to numpy)
        pil = Image.fromarray((arr * 255).astype(np.uint8))
        pil = pil.rotate(random.uniform(-MAX_ROTATION_DEG, MAX_ROTATION_DEG),
                         resample=Image.BICUBIC, fillcolor=255)
        arr = np.array(pil, dtype=np.float32) / 255.0
        del pil

        # Scale (PIL, converted immediately to numpy)
        scale    = random.uniform(0.80, 1.20)
        new_size = max(4, int(size * scale))
        pil2     = Image.fromarray((arr * 255).astype(np.uint8)).resize(
                       (new_size, new_size), Image.LANCZOS)
        small    = np.array(pil2, dtype=np.float32) / 255.0
        del pil2
        canvas   = np.ones((size, size), dtype=np.float32)
        off      = (size - new_size) // 2
        sy, sx   = max(0, off), max(0, off)
        ey       = min(sy + small.shape[0], size)
        ex       = min(sx + small.shape[1], size)
        canvas[sy:ey, sx:ex] = small[:ey - sy, :ex - sx]
        arr      = canvas

        # Flip (numpy, no alloc)
        if random.random() > 0.5:
            arr = arr[:, ::-1].copy()

        # Noise (numpy)
        arr += np.random.normal(0, 0.03, arr.shape).astype(np.float32)
        results.append((np.clip(arr, 0.0, 1.0).astype(np.float32), orig_w, orig_h))
    return results


# Auto-scale augmentation; pre-allocate X — one contiguous block, never doubled
AUGMENT_PER_IMAGE = max(3, min(40, TARGET_TOTAL_SAMPLES // max(1, len(paths))))
total_alloc       = AUGMENT_PER_IMAGE * len(paths)
mem_gb            = total_alloc * FEATURE_DIM * 4 / 1e9

print(f'Base images        : {len(paths):,}')
print(f'Augmentation       : {AUGMENT_PER_IMAGE} variants → ~{total_alloc:,} samples')
print(f'Pre-allocated X    : {mem_gb:.2f} GB')
print('Processing...\n')

X = np.empty((total_alloc, FEATURE_DIM), dtype=np.float32)
y = np.empty(total_alloc,                dtype=np.int32)
idx          = 0
report_every = max(1, len(paths) // 20)

for i, (filepath, label) in enumerate(zip(paths, labels)):
    # Periodic GC + malloc_trim: return freed small allocations to the OS
    if i > 0 and i % 2000 == 0:
        _malloc_trim()
        pct = 100 * i // len(paths)
        print(f'  {pct:3d}%  ({i:,}/{len(paths):,})  [GC done]', flush=True)
    elif i % report_every == 0:
        pct = 100 * i // len(paths)
        print(f'  {pct:3d}%  ({i:,}/{len(paths):,})', flush=True)

    with Image.open(filepath) as img:
        augs = augment_to_array(img, AUGMENT_PER_IMAGE)

    for (arr, orig_w, orig_h) in augs:
        if idx < total_alloc:
            X[idx] = extract_features(arr, orig_w, orig_h)
            y[idx] = label
            idx += 1
    del augs

X = X[:idx]
y = y[:idx]
_malloc_trim()
print(f'\n✅ {idx:,} samples  X: {X.shape}  ({X.nbytes / 1e9:.2f} GB)')

## Step 7 — Train MLP on HOG features

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=42
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(FEATURE_DIM,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
], name='figurine_hog_mlp')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=12, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=6, min_lr=1e-6, monitor='val_accuracy'),
    ],
    verbose=1,
)
print(f'\n✅ Best val accuracy: {max(history.history["val_accuracy"]):.4%}')

## Step 8 — Per-class accuracy report

In [ ]:
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

y_pred_probs = model.predict(X_val, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

print('Per-class spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    probs = y_pred_probs[mask][0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy');  ax1.legend()
ax2.plot(history.history['loss'],         label='train')
ax2.plot(history.history['val_loss'],     label='val')
ax2.set_title('Loss');      ax2.legend()
plt.tight_layout(); plt.show()

## Step 9 — Export TFLite model

In [ ]:
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'✅ Saved: {TFLITE_PATH} ({size_kb:.0f} KB)')

interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f'   Input : {inp["shape"]}  dtype={inp["dtype"].__name__}')
print(f'   Output: {out["shape"]}  dtype={out["dtype"].__name__}')

print('\nTFLite spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    sample = X_val[mask][0:1].astype(np.float32)
    interp.set_tensor(inp['index'], sample)
    interp.invoke()
    probs = interp.get_tensor(out['index'])[0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

## Step 10 — Download model

In [ ]:
from google.colab import files
files.download(TFLITE_PATH)
print(f'✅ Downloaded {TFLITE_PATH}')